# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

All entities—record sets, fields, columns—are referenced by their `@id` to ensure clarity and reproducibility.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
# Display name and description
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Explore available record sets and their fields, referencing each entity by its `@id`.

Let's enumerate record sets, their names and contained fields. This prepares for record loading and analysis.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets
print('Available RecordSets:')
for rs in record_sets:
    print(f'  - @id: {rs.id}, name: {rs.name}')
    print('    Fields:')
    for field in rs.fields:
        print(f'      * @id: {field.id}, name: {field.name}')
    print('    Columns:')
    for col in rs.columns:
        print(f'      * @id: {col.id}, name: {col.name}')
    print()

## 3. Data Extraction
Use the `mlcroissant` API to load data from a specific record set, referencing by its `@id`.

Below, we extract records for each record set found previously and store as DataFrames for analysis.

In [ ]:
# Collect all available record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dfs = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    # Convert to DataFrame for easier analysis
    dfs[rsid] = pd.DataFrame(records)

# If record sets are available, show the columns and preview for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f'RecordSet @id: {first_rs_id}')
    print('Columns:', dfs[first_rs_id].columns.tolist())
    dfs[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data cleaning and transformation operations.

### Steps:
- Filter records based on a numeric column (e.g., GAD-7 score or participant age)
- Normalize the column
- Group by a demographic field and examine aggregate statistics

**All references use the entity `@id` from metadata. Please adjust field names as per record set field listing above.**

In [ ]:
# Choose a record set, numeric field and grouping field by @id
# Example recommendation: use GAD-7 score, PHQ-9 score, or age (field @ids listed previously)

# For demonstration, select the first record set if available
if record_set_ids:
    rsid = record_set_ids[0]
    df = dfs[rsid]
    # Find a numeric field by @id
    numeric_candidates = [col for col in df.columns if 'gad7_score' in col or 'phq9_score' in col or 'age' in col]
    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]
    
    print(f'Using numeric field @id: {numeric_field}')
    
    # Filtering: only values greater than threshold
    threshold = 10
    filtered = df[df[numeric_field] > threshold]
    print(f'Filtered records for {numeric_field} > {threshold}:')
    print(filtered.head())

    # Normalization
    filtered[f'{numeric_field}_normalized'] = (filtered[numeric_field] - filtered[numeric_field].mean()) / filtered[numeric_field].std()
    print(f'Normalized {numeric_field} for filtered records:')
    print(filtered[[numeric_field, f'{numeric_field}_normalized']].head())

    # Grouping: group by demographic field such as gender or education
    group_candidates = [col for col in df.columns if 'gender' in col or 'education' in col]
    group_field = group_candidates[0] if group_candidates else df.columns[0]
    print(f'Grouping by field @id: {group_field}')
    if group_field in filtered.columns:
        grouped = filtered.groupby(group_field)[numeric_field].mean()
        print(f'Grouped data by {group_field}:')
        print(grouped.head())

## 5. Visualization
Visualize the distribution of a numeric field (e.g., GAD-7 score, PHQ-9 score, age) and compare across groups. This section demonstrates histograms and boxplots using pandas and matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Visualize the numeric field distribution
if record_set_ids:
    rsid = record_set_ids[0]
    df = dfs[rsid]
    numeric_candidates = [col for col in df.columns if 'gad7_score' in col or 'phq9_score' in col or 'age' in col]
    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]

    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=20, kde=True, color='teal')
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Compare by group field
    group_candidates = [col for col in df.columns if 'gender' in col or 'education' in col]
    group_field = group_candidates[0] if group_candidates else df.columns[0]
    if group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} (@id) by {group_field} (@id)')
        plt.show()

## 6. Conclusion
This notebook showcased how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using `mlcroissant`, referencing all entities by their `@id`.

- Metadata and structure were reviewed using `mlcroissant` API.
- Record sets and fields were enumerated by their `@id`.
- Numeric analysis and visualizations demonstrated key trends in mental health scores and demographics.

For advanced analyses, reference additional record sets and fields by `@id`, and extend the processing pipeline as suited for your research or application.